In [52]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,SampleID,Age,Avg Monthly GB Download,Avg Monthly Long Distance Charges,City,CLTV,Contract,Country,Customer ID,Dependents,...,Total Revenue,Under 30,Unlimited Data,Zip Code,Ping Score,Avg Speed,TV Type,Link Quality Index,Total Short Distance Charges,Churn
0,1,36,29,0.00,Snelling,4840,Month-to-Month,United States,6581-NQCBA,0,...,810.20,0,1,95369,73.483997,98.441872,Satellite Receiver,40.948442,22.976536,1
1,2,73,27,22.50,San Diego,3208,Month-to-Month,United States,8000-REIQB,0,...,102.45,0,0,92117,49.101754,99.903358,Internet,56.569219,-0.087549,1
2,3,60,13,1.27,Coarsegold,5355,One Year,United States,3442-ZHHCC,0,...,5109.36,0,1,93614,38.435240,99.877390,Satellite Receiver,63.179464,102.365883,0
3,4,54,21,0.00,Fairfax,5097,Month-to-Month,United States,2346-DJQTB,0,...,NaN,0,0,94930,42.155569,100.916879,Satellite Receiver,14.727121,55.686300,0
4,5,54,0,28.73,Myers Flat,2865,Month-to-Month,United States,4760-XOHVN,1,...,48.18,0,0,95554,51.993399,100.523227,Satellite Receiver,18.889887,5.972805,1


In [53]:
#subtasks

def get_financial_score(row):
    score = (row["Monthly Charge"] > 70) + (row["Total Extra Data Charges"] > 10)
    return score

subtask1 = df_test.apply(get_financial_score, axis=1)

df_1 = pd.DataFrame({
    "id": df_test.SampleID,
    "subtaskID": 1,
    "answer": subtask1
})

df_1.head()

,id,subtaskID,answer
0,5635,1,0
1,5636,1,1
2,5637,1,0
3,5638,1,1
4,5639,1,0


In [54]:
def get_service_score(row):
    score = (row["Avg Speed"] < 50) + (row["Ping Score"] > 80) + (row["Link Quality Index"] < 30)
    return score

subtask2 = df_test.apply(get_service_score, axis=1)

df_2 = pd.DataFrame({
    "id": df_test.SampleID,
    "subtaskID": 2,
    "answer": subtask2
})

df_2.head()

,id,subtaskID,answer
0,5635,2,0
1,5636,2,0
2,5637,2,1
3,5638,2,0
4,5639,2,1


In [55]:
#mean mode imputing, label encoding
to_imp_cat = ["Internet Type", "Offer"]
to_imp_num = ["Total Charges", "Total Long Distance Charges", "Total Revenue"]

for col in to_imp_cat:
    df_train[col] = df_train[col].fillna(df_train[col].mode()[0])
    df_test[col] = df_test[col].fillna(df_test[col].mode()[0])

for col in to_imp_num:
    df_train[col] = df_train[col].fillna(df_train[col].mean())

from sklearn.preprocessing import LabelEncoder
cat_cols = [col for col in df_test.columns if df_test[col].dtype == 'object']
for col in cat_cols:
    enc = LabelEncoder()
    enc.fit(pd.concat([df_train[col], df_test[col]]))
    df_train[col] = enc.transform(df_train[col])
    df_test[col] = enc.transform(df_test[col])

df_train.head()

,SampleID,Age,Avg Monthly GB Download,Avg Monthly Long Distance Charges,City,CLTV,Contract,Country,Customer ID,Dependents,...,Total Revenue,Under 30,Unlimited Data,Zip Code,Ping Score,Avg Speed,TV Type,Link Quality Index,Total Short Distance Charges,Churn
0,1,36,29,0.00,928,4840,0,0,4662,0,...,810.200000,0,1,95369,73.483997,98.441872,2,40.948442,22.976536,1
1,2,73,27,22.50,854,3208,0,0,5678,0,...,102.450000,0,0,92117,49.101754,99.903358,1,56.569219,-0.087549,1
2,3,60,13,1.27,204,5355,1,0,2405,0,...,5109.360000,0,1,93614,38.435240,99.877390,2,63.179464,102.365883,0
3,4,54,21,0.00,313,5097,0,0,1610,0,...,3092.766232,0,0,94930,42.155569,100.916879,2,14.727121,55.686300,0
4,5,54,0,28.73,652,2865,0,0,3342,1,...,48.180000,0,0,95554,51.993399,100.523227,2,18.889887,5.972805,1


In [58]:
features = [col for col in df_test.columns if col != 'SampleID']
target = 'Churn'

X = df_train[features]
y = df_train[target]
X_final = df_test[features]

X.head()

,Age,Avg Monthly GB Download,Avg Monthly Long Distance Charges,City,CLTV,Contract,Country,Customer ID,Dependents,Device Protection Plan,...,Total Refunds,Total Revenue,Under 30,Unlimited Data,Zip Code,Ping Score,Avg Speed,TV Type,Link Quality Index,Total Short Distance Charges
0,36,29,0.00,928,4840,0,0,4662,0,0,...,0.0,810.200000,0,1,95369,73.483997,98.441872,2,40.948442,22.976536
1,73,27,22.50,854,3208,0,0,5678,0,0,...,0.0,102.450000,0,0,92117,49.101754,99.903358,1,56.569219,-0.087549
2,60,13,1.27,204,5355,1,0,2405,0,1,...,0.0,5109.360000,0,1,93614,38.435240,99.877390,2,63.179464,102.365883
3,54,21,0.00,313,5097,0,0,1610,0,0,...,0.0,3092.766232,0,0,94930,42.155569,100.916879,2,14.727121,55.686300
4,54,0,28.73,652,2865,0,0,3342,1,0,...,0.0,48.180000,0,0,95554,51.993399,100.523227,2,18.889887,5.972805


In [80]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_val_score

model = CatBoostClassifier(silent=True)

cross_val_score(model, X, y, cv=3, scoring='roc_auc')

array([0.99333421, 0.99262557, 0.99456522])

In [81]:
model.fit(X, y)
y_preds = model.predict_proba(X_final)

In [82]:
df_preds = pd.DataFrame({
    "id": df_test.SampleID,
    "subtaskID": 3,
    "answer": y_preds[:, 1]
})

df_preds.head()

,id,subtaskID,answer
0,5635,3,0.000404
1,5636,3,0.000433
2,5637,3,0.997845
3,5638,3,0.004387
4,5639,3,0.575597


In [83]:
submission = pd.concat([df_1, df_2, df_preds])
submission.to_csv("submission.csv", index=False)